# Готовые представления и классификация обращений

На этот раз текст будет преобразовывать предобученный rubert-tiny2. Мы посмотрим, что возвращает его токенизатор и энкодер, получим по одному вектору на обращение и обучим знакомую логистическую регрессию. Затем сравним её ошибки с символьным TF-IDF.

In [ ]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Загрузка энкодера

Используем модель и токенизатор из подготовленного комплекта. Веса энкодера останутся фиксированными. Первую специальную позицию CLS возьмём по рекомендации автора модели; далее будем проверять её наличие, форму результата и нормировку.

Инструкция по подготовке файлов находится в START. Здесь начнём с одного короткого текста, чтобы увидеть данные, с которыми предстоит работать.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, classification_report

torch.set_num_threads(4)
MODEL_ROOT = Path(os.environ.get("NLP_MODELS_DIR", ROOT / ".cache" / "models"))
MODEL_DIR = MODEL_ROOT / "encoder"
if not (MODEL_DIR / "config.json").exists():
    raise FileNotFoundError("Сначала выполните scripts/prepare_models.py при доступном интернете")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
encoder, loading_info = AutoModel.from_pretrained(MODEL_DIR, local_files_only=True,
                                                 dtype=torch.float32, output_loading_info=True)
if loading_info["missing_keys"] or loading_info["mismatched_keys"] or loading_info["error_msgs"]:
    raise RuntimeError("Некорректная загрузка весов энкодера")
assert all(k.startswith("cls.") for k in loading_info["unexpected_keys"])
encoder.eval(); encoder.requires_grad_(False)
print("Размерность:", encoder.config.hidden_size, "Слоёв:", encoder.config.num_hidden_layers)

df = pd.read_csv(DATA / "requests.csv", keep_default_na=False)
parts = {name: df[df["split"].eq(name)].copy() for name in ["train", "validation", "test"]}
labels = sorted(df["label"].unique())


In [ ]:
demo_batch = tokenizer(["Как продлить книгу?"], return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(demo_batch["input_ids"][0].tolist()))
with torch.inference_mode():
    demo_hidden = encoder(**demo_batch).last_hidden_state
print("Вход:", tuple(demo_batch["input_ids"].shape))
print("Выход:", tuple(demo_hidden.shape))

Первая ось выхода относится к текстам пакета, вторая — к токенам, третья — к 312 координатам скрытого представления. Найдите CLS в списке токенов. В следующем опыте появятся несколько текстов разной длины и padding.

## 1. Токены и длина входа

Для трёх строк из заготовки создайте `token_batch` с `padding=True`, `truncation=True`, `max_length=128`, `return_tensors="pt"`. Выведите формы полей, названия токенов и маску каждой строки. Убедитесь, что первая позиция — ожидаемый CLS.

Для всех обращений корпуса получите полные длины до усечения в массиве `lengths`. Число значений больше 128 сохраните как целое `truncated_count`. Учитывайте специальные позиции. Чем отличается добавление padding от усечения?

In [ ]:
examples = ["Как продлить книгу?", "Не работает личный кабинет.", "Спасибо"]
# Ваш код
raise NotImplementedError

In [ ]:
assert token_batch["input_ids"].shape[0] == 3
assert token_batch["input_ids"].shape == token_batch["attention_mask"].shape
assert (token_batch["input_ids"][:,0] == tokenizer.cls_token_id).all()
assert len(lengths) == len(df) and 0 <= truncated_count <= len(df)

## 2. По одному вектору на текст

Напишите `encode_texts(texts, batch_size=16)`. Обрабатывайте тексты пакетами с прежними настройками токенизатора. Передайте модели все поля пакета, возьмите `hidden[:, 0, :]`, нормируйте по оси признаков `dim=1` и объедините строки в исходном порядке. Используйте `eval()` и `torch.inference_mode()`; веса уже заморожены.

Пустой список должен возвращать массив формы `(0, 312)`. Для непустого результата первая ось равна числу текстов. Создайте `features` — словарь массивов для train, validation, test. Покажите их формы. Что изменится при ошибочной записи `hidden[0, 0, :]`?

In [ ]:
def encode_texts(texts, batch_size=16):
    # Ваш код
    raise NotImplementedError

# Ваш код
raise NotImplementedError

In [ ]:
for name, part in parts.items():
    assert features[name].shape == (len(part), 312)
    assert np.isfinite(features[name]).all()
    assert np.allclose(np.linalg.norm(features[name], axis=1), 1, atol=1e-5)
assert not encoder.training
assert all(not p.requires_grad for p in encoder.parameters())
assert encode_texts([]).shape == (0,312)

## 3. Соседний текст и padding

Сравните «Как продлить книгу?» отдельно и в пакете с более длинным обращением. Запишите максимальную абсолютную разницу в `batch_difference`; она должна быть меньше 1e-5. Повторите отдельное кодирование той же строки.

Для ручной матрицы `H = [[1, 0], [0, 1], [9, 9]]` и маски `[1, 1, 0]` вычислите обычное среднее и `masked_mean`. Почему исключение padding из внимания не позволяет автоматически усреднять все выходные позиции? Основной классификатор продолжает использовать CLS.

In [ ]:
# Ваш код
raise NotImplementedError

In [ ]:
assert batch_difference < 1e-5
assert np.allclose(masked_mean, [0.5,0.5])

## 4. Два представления, один способ классификации

Обучите `embedding_clf=LogisticRegression(C=1,max_iter=1000,random_state=42)` на features["train"]. Классический `lexical` — Pipeline с char_wb(3,5), max_features=5000 и такой же регрессией. Оцените accuracy и macro-F1 по всем 6 меткам на validation/test. Конфигурации не меняются после просмотра чисел. `scores_table` содержит столбцы split, model, accuracy, macro_f1. `predictions` — словарь с ключами "tfidf" и "encoder" и массивами предсказаний теста. Постройте матрицу ошибок для encoder.

Сопоставьте общую accuracy с recall отдельных категорий. Какой класс потерял больше всего обращений?

In [ ]:
# Ваш код
raise NotImplementedError

In [ ]:
assert set(predictions) == {"tfidf", "encoder"}
assert all(len(p) == len(parts["test"]) for p in predictions.values())
assert scores_table["macro_f1"].between(0,1).all()
assert all(not p.requires_grad for p in encoder.parameters())

## 5. Читаем удачные и ошибочные решения

Создайте `error_cases` для всех тестовых строк с колонками `id`, `group_id`, `text`, `label`, `tfidf`, `encoder`. Затем отдельно отберите до пяти разных групп: хотя бы одну ошибку энкодера и одно исправление ошибки TF-IDF, если они есть.

Прочитайте обращения вместе с метками. В чём заключается затруднение: близкие темы, граница организационных категорий, недостаток контекста? Для гипотезы о влиянии слова предложите отдельную проверку. Один удачный или ошибочный пример сам по себе не объясняет механизм модели.

In [ ]:
# Ваш код
raise NotImplementedError

In [ ]:
assert len(error_cases) == len(parts["test"])
assert {"text","label","tfidf","encoder","group_id"} <= set(error_cases.columns)

## Просьба осталась за пределом входа

В основном корпусе усечений не было. Теперь создадим отдельный длинный пример с просьбой в конце. До запуска предскажите, увидит ли модель эту просьбу при прежнем max_length. После усечения прочитайте сохранённый текст.

In [ ]:
# Длинный диагностический вход: важная просьба только в конце.
long_text = "Уточняю ранее отправленный вопрос. " * 80 + "Как продлить книгу?"
full_ids = tokenizer(long_text, truncation=False)["input_ids"]
cut_ids = tokenizer(long_text, truncation=True, max_length=128)["input_ids"]
cut_text = tokenizer.decode(cut_ids, skip_special_tokens=True)
print("Полная длина:", len(full_ids), "После усечения:", len(cut_ids))
print("Конец после усечения:", cut_text[-150:])
print("Слово 'книгу' сохранилось:", "книгу" in cut_text)
assert len(full_ids)>128 and len(cut_ids)==128 and "книгу" not in cut_text


## Длина, сущности и выводы

Объясните, какая часть длинного диагностического обращения осталась на входе модели и что это означает для классификации. Разметьте BIO-метками слова «Анна Петрова учится в университете Север»: имя включает первые два слова, организация — последние два. Если система найдёт только «Петрова», будет ли это правильной полной сущностью при строгом совпадении границ?

Завершите ноутбук разбором метрик, одной ошибки энкодера и одного улучшения относительно TF-IDF. Перед сдачей перезапустите ядро и выполните все ячейки по порядку.